# EDA — before any inference

Goal: understand the dataset and catch problems *before* running regressions.
Run `python src/make_dataset.py` (or `--sample`) first.

Questions to answer here (Phase 4 of ROADMAP.md):
1. How many player-seasons? How many are contract years?
2. Do contract years skew older? (This is why we control for age.)
3. Is the outcome metric sane — any impossible values?
4. Where is data missing, and does it look random?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv(Path('..') / 'data' / 'processed' / 'player_seasons.csv')
print(f"{len(df)} player-seasons, {df.player_id.nunique()} players, "
      f"seasons {df.season.min()}-{df.season.max()}")
df.head()

In [ ]:
# 1. counts and contract-year share
print(df.contract_year.value_counts())
print(f"\ncontract-year share: {df.contract_year.mean():.1%}")

In [ ]:
# 2. age distribution by contract status — expect contract years to skew older
fig, ax = plt.subplots(figsize=(8, 4))
df[df.contract_year == 0].age.hist(bins=range(18, 42), alpha=0.6,
                                   label='non-contract', ax=ax, density=True)
df[df.contract_year == 1].age.hist(bins=range(18, 42), alpha=0.6,
                                   label='contract year', ax=ax, density=True)
ax.set_xlabel('age'); ax.set_ylabel('density'); ax.legend()
ax.set_title('Age distribution: contract vs non-contract seasons')
plt.show()
df.groupby('contract_year').age.describe()

In [ ]:
# 3. outcome sanity — distribution and extremes
fig, ax = plt.subplots(figsize=(8, 4))
df.bpm.hist(bins=40, ax=ax)
ax.set_xlabel('bpm'); ax.set_title('Outcome distribution')
plt.show()
print('most extreme seasons:')
df.nlargest(5, 'bpm')[['player_name', 'season', 'age', 'bpm', 'contract_year']]

In [ ]:
# 4. missingness
df.isna().sum().to_frame('missing_cells').query('missing_cells > 0')

In [ ]:
# naive raw comparison — WRONG design, shown to make the point:
# this difference mixes the real effect with age/selection bias.
# The FE regression (Phase 5) is what isolates the within-player effect.
df.groupby('contract_year').bpm.agg(['mean', 'std', 'count'])